Mit Colab kann **dieses Jupyter-Notebook interaktiv im Browser** gestartet werden:
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/DonGoginho/myPy/blob/main/parlamentsdienste_ris_api/pd_ris_api_al_geschaefte.ipynb)

Und hier ein Jupyter-Notebook zur [generellen Verwendung der RIS-API](https://colab.research.google.com/github/opendatazurich/opendatazurich.github.io/blob/master/ris-api/RIS-API-Beispiele.ipynb)
# Python-Abfrage an RIS-API nach Geschäftern, die von der Alternativen Liste eingegeben wurden

Das Datenmodell der [**RIS-API ist als Swagger File**](https://opendatazurich.github.io/ris-api/docs/) verfügbar. Für das Verständnis ist dieses daher wichtig. 

**Inhaltsverzeichnis:**

1. [Mitglieder der Alternativen Liste](#Mitglieder-der-AL)
1. [Gemeinderats-Geschäfte](#Gemeinderats-Geschäfte)

In [1]:
%pip install requests pandas

Note: you may need to restart the kernel to use updated packages.


In [2]:
import requests
from pprint import pprint
import pandas as pd
import random

In [3]:
SSL_VERIFY = False
# evtl. SSL_VERIFY auf False setzen wenn die Verbindung zu https://www.gemeinderat-zuerich.ch nicht klappt (z.B. wegen Proxy)
# Um die SSL Verifikation auszustellen, bitte die nächste Zeile einkommentieren ("#" entfernen)
# SSL_VERIFY = False

In [4]:
if not SSL_VERIFY:
    import urllib3
    urllib3.disable_warnings()

## Infos zu AL
- 'parteiId': 'df5ccd54-009e-40ae-b687-4ce88a71f5d8'
- 'fraktionsId': '53bf7290-4055-455b-bb1f-2ba64e76294d'

## Mitglieder der AL

Um alle Mitglieder einer Partei zu bekommen, müssen folgende Abfragen gemacht werden:
    
    1. ID der Partei holen
    1. Suche nach Mitgliedern der Partei (mit dieser ID)
    1. *Optional:* Detail-Informationen eines Mitglieds holen
    
    

### Alle relevanten Mitglieder-Parameter holen

Lese alle Parameter zu den Mitgliedern in `params` ein

In [5]:
headers = {'Accept': 'application/json'}
r = requests.get('https://www.gemeinderat-zuerich.ch/api/Mitglieder/parameter', headers=headers, verify=SSL_VERIFY)
params = r.json()
#params

Lese die Informationen zu den Parteien aus `params`heraus

In [6]:
# get id of AL
parties = params['Parteien']
#party = {'parteiId': 'df5ccd54-009e-40ae-b687-4ce88a71f5d8'}
parties

[{'Name': 'AL', 'Id': 'df5ccd54-009e-40ae-b687-4ce88a71f5d8'},
 {'Name': 'CVP', 'Id': '5964b653-a32a-4e0c-92a1-07720566a92c'},
 {'Name': 'EVP', 'Id': '187b7ad1-2095-43cc-b257-e9834a54a961'},
 {'Name': 'FDP', 'Id': '8240033e-4818-4406-8f73-b7829d99ddee'},
 {'Name': 'GLP', 'Id': '0a8eb90f-6d09-47ac-99fe-ab97a77d4954'},
 {'Name': 'Grüne', 'Id': 'fe956d66-e14d-4739-82cf-c12202d96445'},
 {'Name': 'Parteilos', 'Id': '9e037de7-40a4-4fa3-85b7-f4c30cbb26c8'},
 {'Name': 'SP', 'Id': '6cb1a3bb-8e26-4818-a8f5-0ea49ab3cd57'},
 {'Name': 'SVP', 'Id': '8e484ad9-6e46-4d39-bc05-fbf18be11b0c'}]

Für diese Anfrage ist die AL relevant. Definiere daher als `party`die gesuchte Partei

In [7]:
party = {'parteiId': 'df5ccd54-009e-40ae-b687-4ce88a71f5d8','Name': 'AL'}
party

{'parteiId': 'df5ccd54-009e-40ae-b687-4ce88a71f5d8', 'Name': 'AL'}

### Mitglieder der AL abfragen

In der Query wird die ParteiId angegeben, welche für die Abfrage des Endpoints relevant ist.
In `members`wird das Resultat der Abfrage als JSON-Array eingelesen.

In [8]:
query = {'parteiId':  'df5ccd54-009e-40ae-b687-4ce88a71f5d8'}
#query = {'fraktionsId':  '53bf7290-4055-455b-bb1f-2ba64e76294d'}
headers = {'Accept': 'application/json'}
r = requests.get('https://www.gemeinderat-zuerich.ch/api/Mitglieder/suchen', params=query, headers=headers, verify=SSL_VERIFY)
members = r.json()
#members

### Details zu den AL-Mitgliedern abfragen

Mit dem `/Mitglieder/details` Endpunkt können Details eines Mitglieds abgefragt werden. Dazu muss die Mitglieder-ID jedes Mitglieds herausgelesen und über die API abgefragt werden. Dies erfolgt über die Funktion `get_member_details()`.

In [9]:
#members

In [10]:
def get_member_details(member):
    #print(member)
    query = {'mid': member['Id']}
    #print(query)
    headers = {'Accept': 'application/json'}
    r = requests.get('https://www.gemeinderat-zuerich.ch/api/Mitglieder/details', params=query, headers=headers, verify=SSL_VERIFY)
    details = r.json()
    
    # trim "Wahlkreis" and "Wohnkreis" since they have a padding
    details['Wahlkreis'] = details['Wahlkreis'].strip()
    if details['Wohnkreis']:
        details['Wohnkreis'] = details['Wohnkreis'].strip()
    # add original member to keep WahlkreisOrderBy column
    details.update(member)
    return details
    
member_details = [get_member_details(m) for m in members]
df = pd.DataFrame(member_details)
df.head(3)

,Id,Name,Vorname,Anrede,Titel,Geburtstag,Wohnkreis,Beruf,Partei,Fraktion,...,EmailPrivat,EmailGeschaeftlich,Mobiltelefon,MobiltelefonGeschaeftlich,TelefonGeschaeftlich,TelefonPrivat,Internetauftritt,Interessenverbindungen,NameInUrl,WahlkreisOrderBy
0,ee8204f4-9290-49ca-99ae-d18a22a0a513,Angst,Walter,Herr,None,1961-09-22T00:00:00,3,Leiter Kommunikation,AL,AL,...,None,walterangst@gmail.com,079 288 56 92,None,044 296 90 30,None,None,"- Mieterinnen– und Mieterverband Zürich, Leite...",Walter%20Angst,101
1,bbfb8ca0-91b1-448d-b1fc-c5e3bb35c158,Eberle,Natalie,Frau,None,1967-04-24T00:00:00,9,Soziale Quartierentwicklung,AL,AL,...,,quartiernetz@qnf.ch,,None,043 537 80 17,,None,"- Quartiernetz Friesenberg, Stellenleitung",Natalie%20Eberle,101
2,9e60ce03-30ae-4e51-be43-28e81fcbf0f6,Fischer Svosve,Regula,Frau,None,1964-01-03T00:00:00,10,"lic. phil. I, Mitarbeiterin Administration",AL,AL,...,regulaf@gmail.com,None,,None,None,044 271 42 74,None,"- Verein Wartsaal Wipkingen, Zürich, Präsidentin",Regula%20Fischer%20Svosve,106


In [11]:
# Beispiel einer Grafik dazu
#count_by_wahlkreis = df[['Wahlkreis', 'WahlkreisOrderBy']].groupby(['Wahlkreis', 'WahlkreisOrderBy']).size().reset_index(name='Anzahl')
#count_by_wahlkreis = count_by_wahlkreis.sort_values(by=['WahlkreisOrderBy'])
#count_by_wahlkreis.plot.bar(x='Wahlkreis', y='Anzahl')

# Anfrage der Parlamentsdienste

Ziel dieser Anfrage ist es eine Liste zu erhalten, welche alle Geschäfte aufführt, welche seitens AL-Ratsmitglied oder der AL-Fraktion jemals eingereicht wurden. Dies sollten rund 660 Geschäfte sein.

Vorgegebene Bedingungen:
- Geschäftsart=Motion, Postulat, Interpellation, Schriftliche Anfrage, Beschlussantrag oder Globalbudgetantrag
- Akteure/Einreichende Person=AL (muss der Partei "AL" angehören)
- Zeitraum=unbegrenzt 

IDs zur AL
- 'parteiId': 'df5ccd54-009e-40ae-b687-4ce88a71f5d8'
- 'fraktionId': '53bf7290-4055-455b-bb1f-2ba64e76294d'

Resultat: 
Liste mit Angaben zu
- Geschäftsnummer
- Geschäftstitel
- Geschäftsart
- Einreichende Person/Partei/Fraktion: AL
- Link auf das Geschäft auf der Website des Gemeinderats



## Gemeinderats-Geschäfte die von der AL eingereicht wurden

Im der Resultatsliste sollen alls Geschäfte aufgelistet werden, welche die AL entweder als Partei oder als Fraktion eingegeben hat.

1.   Stelle zuerst den Request auf die Geschäfte, welche von der AL **als Fraktion** eingegeben wurden und übergebe das Resultat aus einem JSON (`result_geschaefte_fraktion`) in einen Dataframe (`df_deals_fraktion`).

In [12]:
#query = {'personId': member['Id']}

# via swagger: URL mit einem OR Operatoren drin
#https://www.gemeinderat-zuerich.ch/api/Geschaeft/suchen?parteiId=df5ccd54-009e-40ae-b687-4ce88a71f5d8|fraktionId=53bf7290-4055-455b-bb1f-2ba64e76294d&orderBy=geschaeftsart

query_fraktion = {'fraktionId': '53bf7290-4055-455b-bb1f-2ba64e76294d', 'activePage':1, 'pageSize':700}
headers = {'Accept': 'application/json'}
r = requests.get('https://www.gemeinderat-zuerich.ch/api/Geschaeft', 
                 params=query_fraktion, 
                 headers=headers, 
                 verify=SSL_VERIFY)
result_geschaefte_fraktion = r.json()
df_deals_fraktion = pd.DataFrame(result_geschaefte_fraktion['Geschaefte'])
print('done')




done


In [13]:
df_deals_fraktion.info

<bound method DataFrame.info of                               GeschaeftId  Geschaeftsjahr  Geschaeftsnummer  \
0    56b6f886-ceca-4774-9fd8-24b445267ebe            2022               126   
1    b21dde90-7652-44e6-bc1f-abf23b760d12            2022               103   
2    dd8df229-376d-457a-a2fc-239bd62bf5a8            2022                77   
3    42292e92-ca3b-491e-933d-1b80c062666f            2022                70   
4    49e308c3-95f4-4c09-89b7-d001cc58f5c5            2022                26   
..                                    ...             ...               ...   
365  c6cdb895-9dac-4d46-bd5a-ffcc5a903f75            2006               245   
366  9b42e101-2c32-43d9-a86a-c76587659016            2006               231   
367  278d6994-fd9b-451f-bae8-3e1395ecd24e            2006               177   
368  eda7ab1d-227c-40f1-9e75-ebe0fb21516f            2006               162   
369  14ebe587-f8c4-436c-9deb-595929305b8c            2006               158   

                   



2.   Stelle anschliessend den Request auf die Geschäfte, welche von der AL **als Partei** eingegeben wurden und übergebe das Resultat aus einem JSON (result_geschaefte_partei) in einen Dataframe (df_deals_partei).

In [14]:
query_partei = {'parteiId': 'df5ccd54-009e-40ae-b687-4ce88a71f5d8', 'activePage':1, 'pageSize':700}
headers = {'Accept': 'application/json'}
r = requests.get('https://www.gemeinderat-zuerich.ch/api/Geschaeft', 
                 params=query_partei, 
                 headers=headers, 
                 verify=SSL_VERIFY)
result_geschaefte_partei = r.json()
df_deals_partei = pd.DataFrame(result_geschaefte_partei['Geschaefte'])
print('done')


done


In [15]:
df_deals_partei.info

<bound method DataFrame.info of                               GeschaeftId  Geschaeftsjahr  Geschaeftsnummer  \
0    b481cb8a-14e0-480e-9a6d-5ad91e1e7fc7            2022               109   
1    8783c40d-75fc-4b61-9727-5c643830865d            2022               105   
2    14308906-9dca-4dba-8d76-6642c46c2041            2022                91   
3    10d92e19-b575-4c8c-b13d-a2fe73bd15f8            2022                49   
4    59be654f-3cbe-4037-99d1-a9c3563f6193            2022                48   
..                                    ...             ...               ...   
625  1afa9df6-8635-480d-a8b9-b8214b7c6613            1997               285   
626  c928d093-0db6-4eaa-b30c-6196035b99e5            1996               490   
627  055190a3-536d-4dbe-97fb-a39677fc57d9            1996               100   
628  09ece0e7-4efc-4fcc-a825-847157382b42            1994               424   
629  94afa6b8-e776-455b-b2d9-bb34975c5f47            1994                 8   

                   

Füge nun beide DataFrames der Teilanfragen (1 und 2) zusammen. 
`ignore_index` ist hier wichtig, sonst funktioniert das zusammenhängen nicht wie erwünscht.
Sortiere danach den DataFrame absteigend nach Geschäftsjahr und Geschäftsnummer und schreibe ihn in den DataFrame df_deals.


In [16]:
df_deals_appended = df_deals_partei.append(df_deals_fraktion, verify_integrity=True, ignore_index=True, sort=True)
df_deals = df_deals_appended.sort_values(by=['Geschaeftsjahr', 'Geschaeftsnummer'], ascending=False)
df_deals
#result = df1.append(df4, sort=False)

,GeschaeftId,Geschaeftsart,Geschaeftsjahr,Geschaeftsnummer,Geschaeftstitel
630,56b6f886-ceca-4774-9fd8-24b445267ebe,Postulat,2022,126,Ausrichtung einer Energiezulage an einkommenss...
0,b481cb8a-14e0-480e-9a6d-5ad91e1e7fc7,Schriftliche Anfrage,2022,109,"Sanierung des Gebäudes am Hechtplatz, Planungs..."
1,8783c40d-75fc-4b61-9727-5c643830865d,Postulat,2022,105,"Personalhochhäuser auf dem Triemli-Areal, Über..."
631,b21dde90-7652-44e6-bc1f-abf23b760d12,Beschlussantrag,2022,103,Rückkehr des Gemeinderats in das Rathaus
2,14308906-9dca-4dba-8d76-6642c46c2041,Postulat,2022,91,Ermöglichung eines Steuerabzugs vom Lohn für d...
...,...,...,...,...,...
625,1afa9df6-8635-480d-a8b9-b8214b7c6613,Motion,1997,285,"Behördenmitglieder, Neuregelung der Entschädig..."
626,c928d093-0db6-4eaa-b30c-6196035b99e5,Postulat,1996,490,"Zoo Erweiterung, Verlängerung der Tramlinie 6 ..."
627,055190a3-536d-4dbe-97fb-a39677fc57d9,Postulat,1996,100,"Pumpspeicherwerk Grimsel West, Verhinderung we..."
628,09ece0e7-4efc-4fcc-a825-847157382b42,Postulat,1994,424,"Schulbehörden, Neufestlegung der Strukturen"


In [17]:
df_deals = df_deals_appended.sort_values(by=['Geschaeftsjahr', 'Geschaeftsnummer'], ascending=False)


Check Output des Dataframes

In [18]:
df_deals.head()

,GeschaeftId,Geschaeftsart,Geschaeftsjahr,Geschaeftsnummer,Geschaeftstitel
630,56b6f886-ceca-4774-9fd8-24b445267ebe,Postulat,2022,126,Ausrichtung einer Energiezulage an einkommenss...
0,b481cb8a-14e0-480e-9a6d-5ad91e1e7fc7,Schriftliche Anfrage,2022,109,"Sanierung des Gebäudes am Hechtplatz, Planungs..."
1,8783c40d-75fc-4b61-9727-5c643830865d,Postulat,2022,105,"Personalhochhäuser auf dem Triemli-Areal, Über..."
631,b21dde90-7652-44e6-bc1f-abf23b760d12,Beschlussantrag,2022,103,Rückkehr des Gemeinderats in das Rathaus
2,14308906-9dca-4dba-8d76-6642c46c2041,Postulat,2022,91,Ermöglichung eines Steuerabzugs vom Lohn für d...


Exportiere die Geschäfte dieser Anfrage als Excel und als CSV.


In [19]:
#df_deals.to_csv('deals.csv')  
#df_deals.to_excel('deals.xlsx')  

## Details der Gemeinderats-Geschäfte abfragen

Die Abfrage über den Geschäftsendpoint ('https://www.gemeinderat-zuerich.ch/api/Geschaeft') liefert keine detaillierten Daten. Z.B wer das Geschäft eingereicht hat oder wie der aktuelle Stand dabei ist.

Dazu muss der Endpoint zu den Geschäftsdetails ('https://www.gemeinderat-zuerich.ch/api/Geschaeft/details') abgefragt werden. Dieser benötigt als Input die ID des Geschäfts (die gid).


In [20]:
def get_geschaefte_details(index):
    #Herauslesen der GeschaeftId aus dem Dataframe df_deals
    query = {'gid': df_deals['GeschaeftId'][index]}
    #print(query)
    headers = {'Accept': 'application/json'}
    r = requests.get('https://www.gemeinderat-zuerich.ch/api/Geschaeft/details', params=query, headers=headers, verify=SSL_VERIFY) 
    details = r.json()
    # Da komischerweise in den Details die gid nicht mehr vorkommt, weise ich sie hier explizit zu, damit sie im Output vorkommt.
    details['gid'] = df_deals['GeschaeftId'][index]
    return details
 
#Iterrows ist nicht die schnellste Variante der Abfrage. Aber hier dennoch zweckmässig.
deals_details = [get_geschaefte_details(index) for index, row in df_deals.iterrows()]
print("done")




done


Einlsesen des JSON-Outputs in einen DataFrame.
Beachte die Struktur des Outputs. Siehe auch https://opendatazurich.github.io/ris-api/docs/#/Geschaeft/get_Geschaeft_details > Model

In [21]:
df_deals_details = pd.DataFrame(deals_details)
df_deals_details



,GeschaeftTitel,GeschaeftsJahr,GeschaeftsNummer,GeschaeftsArt,Erstunterzeichnender,ZweiterErstunterzeichnender,EinreichendeGruppe,PendentBei,FristBis,Departemente,Ablaufschritte,VerknuepfteGeschaefte,gid
0,Ausrichtung einer Energiezulage an einkommenss...,2022,126,Postulat,None,None,"[{'Name': 'AL', 'Id': '53bf7290-4055-455b-bb1f...",Stadtrat,2022-07-06T21:13:53.573,"[{'Id': 1, 'Name': 'Zuteilung ausstehend'}]","[{'Datum': '2022-04-06T00:00:00', 'Created': '...",[],56b6f886-ceca-4774-9fd8-24b445267ebe
1,"Sanierung des Gebäudes am Hechtplatz, Planungs...",2022,109,Schriftliche Anfrage,"{'Id': '2d87f64f-7641-4a88-a11d-b8e4a27f29d7',...","{'Id': 'ddcd41f6-9873-4002-80d8-dfd87114c9a5',...",[],Stadtrat,2022-06-23T00:00:00,"[{'Id': 8, 'Name': 'Hochbaudepartement (HBD)'}]","[{'Datum': '2022-03-23T00:00:00', 'Created': '...",[],b481cb8a-14e0-480e-9a6d-5ad91e1e7fc7
2,"Personalhochhäuser auf dem Triemli-Areal, Über...",2022,105,Postulat,"{'Id': '438fa20f-61c4-49d5-8b76-06bbe37e4d74',...","{'Id': 'ee8204f4-9290-49ca-99ae-d18a22a0a513',...",[],Stadtrat,2022-05-06T00:00:00,"[{'Id': 1, 'Name': 'Zuteilung ausstehend'}]","[{'Datum': '2022-04-06T00:00:00', 'Created': '...",[{'GeschaeftId': '3de6382c-4a2a-411c-82be-ee70...,8783c40d-75fc-4b61-9727-5c643830865d
3,Rückkehr des Gemeinderats in das Rathaus,2022,103,Beschlussantrag,None,None,"[{'Name': 'AL', 'Id': '53bf7290-4055-455b-bb1f...",,0001-01-01T00:00:00,[],"[{'Datum': '2022-04-06T00:00:00', 'Created': '...",[],b21dde90-7652-44e6-bc1f-abf23b760d12
4,Ermöglichung eines Steuerabzugs vom Lohn für d...,2022,91,Postulat,"{'Id': 'd7891845-3c88-4539-9f3a-40c72d2036af',...",None,[],Gemeinderat,None,"[{'Id': 4, 'Name': 'Finanzdepartement (FD)'}]","[{'Datum': '2022-03-23T00:00:00', 'Created': '...",[],14308906-9dca-4dba-8d76-6642c46c2041
...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,"Behördenmitglieder, Neuregelung der Entschädig...",1997,285,Motion,"{'Id': 'c05a85e4-caf9-48dd-9959-ea4707c84065',...","{'Id': 'b37fdd87-03a3-4505-9b39-85823968e10d',...",[],,0001-01-01T00:00:00,"[{'Id': 4, 'Name': 'Finanzdepartement (FD)'}]","[{'Datum': '1998-05-13T00:00:00', 'Created': '...",[],1afa9df6-8635-480d-a8b9-b8214b7c6613
996,"Zoo Erweiterung, Verlängerung der Tramlinie 6 ...",1996,490,Postulat,"{'Id': 'ee86db77-a1a9-4def-8b78-b5e4ecd27c9a',...",None,[],,0001-01-01T00:00:00,"[{'Id': 9, 'Name': 'Departement der Industriel...","[{'Datum': '2009-09-17T00:00:00', 'Created': '...",[],c928d093-0db6-4eaa-b30c-6196035b99e5
997,"Pumpspeicherwerk Grimsel West, Verhinderung we...",1996,100,Postulat,"{'Id': '3b7ede89-f4fd-444b-81a7-53a66560333e',...","{'Id': 'ee86db77-a1a9-4def-8b78-b5e4ecd27c9a',...",[],Stadtrat,None,"[{'Id': 9, 'Name': 'Departement der Industriel...","[{'Datum': '2020-11-25T00:00:00', 'Created': '...",[{'GeschaeftId': 'ef0539fa-2bb4-45a2-9c42-81c2...,055190a3-536d-4dbe-97fb-a39677fc57d9
998,"Schulbehörden, Neufestlegung der Strukturen",1994,424,Postulat,"{'Id': 'e1297508-be1a-4c61-81b6-a8f501114378',...",None,[],,0001-01-01T00:00:00,"[{'Id': 10, 'Name': 'Schul- und Sportdeparteme...","[{'Datum': '2005-03-02T00:00:00', 'Created': '...",[{'GeschaeftId': 'dacb1cc2-3170-4291-baaa-fe99...,09ece0e7-4efc-4fcc-a825-847157382b42


In der Anfrage ist ein Link auf das Geschäft gewünscht. Die URL hat einen fixen Teil an den am Ende die gid des Geschäfts angehängt wird.
Da der Link nicht im Datenmodell vorhanden ist, wird er hier erstellt und dem DataFrame angehängt.


In [22]:
#df_deals_details['joined_col'] = df_deals_details.GeschaeftsArt.str.cat(df_deals_details.PendentBei)
basis_url_geschaefte = "https://www.gemeinderat-zuerich.ch/geschaefte/detailansicht-geschaeft?gid="

df_deals_details['deals_details_url'] = basis_url_geschaefte + df_deals_details.gid

#df_deals_details.head(3)
#print(basis_url_geschaefte +''+ df_deals_details.gid)


Exportiere den erstellten DataFrame.
Z.K. im DataFrame sieht es aus als ob die URL nicht sauber Concatenated wird. Im Excel-Output ist jedoch alles i.O.

In [23]:
df_deals_details.to_excel('deals_details.xlsx')  
#url_geschaefte = "https://www.gemeinderat-zuerich.ch/geschaefte/detailansicht-geschaeft?gId="df_deals_details['gid']

Z.K. hier eine kompliziertere Version wie die URL konstruiert werden kann. Ist aber letztlich nicht nötig mit Pandas...

In [24]:

"""
for index, row in df_deals_details.iterrows():

    url_geschaefte = "https://www.gemeinderat-zuerich.ch/geschaefte/detailansicht-geschaeft?gid="
    #gid=deals_details['gid'][index]
    a=df_deals_details['gid'][index]

    print(url_geschaefte+''+a)
"""

'\nfor index, row in df_deals_details.iterrows():\n\n    url_geschaefte = "https://www.gemeinderat-zuerich.ch/geschaefte/detailansicht-geschaeft?gid="\n    #gid=deals_details[\'gid\'][index]\n    a=df_deals_details[\'gid\'][index]\n\n    print(url_geschaefte+\'\'+a)\n'

In [25]:
df_deals_details.columns


Index(['GeschaeftTitel', 'GeschaeftsJahr', 'GeschaeftsNummer', 'GeschaeftsArt',
       'Erstunterzeichnender', 'ZweiterErstunterzeichnender',
       'EinreichendeGruppe', 'PendentBei', 'FristBis', 'Departemente',
       'Ablaufschritte', 'VerknuepfteGeschaefte', 'gid', 'deals_details_url'],
      dtype='object')

### Normalisierung der Geschäftsdetails

Der Output der Detailinformationen ist verschachtelt, wie im [Datenmodell](https://opendatazurich.github.io/ris-api/docs/#/Geschaeft/get_Geschaeft_details) beschrieben. In den nächsten Schritten wird der DataFrame mit den Datails zu den Geschäften normalisiert. Siehe dazu z.B folgende Links:
- https://pandas.pydata.org/pandas-docs/version/0.21/generated/pandas.io.json.json_normalize.html
- https://towardsdatascience.com/all-pandas-json-normalize-you-should-know-for-flattening-json-13eae1dfb7dd
- https://blog.softhints.com/python-normalize-json-pandas/


Hier im ersten Versuch werden keine weiteren Parameter angegeben. Es ist jedoch ersichtlich, dass dies nicht dem von uns erhofften Output entspricht. Es ist unklar, was mit den Erstunterzeichnenden geschehen ist ... jedenfalls nützt das so nichts.

In [26]:
from pandas.io.json import json_normalize

In [27]:

#df_deals_details_normalized = pd.json_normalize(deals_details, sep="_")

df_deals_details_normalized = pd.json_normalize(deals_details)
#, max_level=1


df_deals_details_normalized.columns
#df_deals_details_normalized.head(3)
#df_deals_details_normalized.to_excel(path+'deals_details_normalized.xlsx')  

Index(['GeschaeftTitel', 'GeschaeftsJahr', 'GeschaeftsNummer', 'GeschaeftsArt',
       'Erstunterzeichnender', 'ZweiterErstunterzeichnender',
       'EinreichendeGruppe', 'PendentBei', 'FristBis', 'Departemente',
       'Ablaufschritte', 'VerknuepfteGeschaefte', 'gid',
       'Erstunterzeichnender.Id', 'Erstunterzeichnender.Name',
       'Erstunterzeichnender.Vorname', 'Erstunterzeichnender.Titel',
       'Erstunterzeichnender.Partei', 'Erstunterzeichnender.Wahlkreis',
       'Erstunterzeichnender.WahlkreisOrderBy',
       'ZweiterErstunterzeichnender.Id', 'ZweiterErstunterzeichnender.Name',
       'ZweiterErstunterzeichnender.Vorname',
       'ZweiterErstunterzeichnender.Titel',
       'ZweiterErstunterzeichnender.Partei',
       'ZweiterErstunterzeichnender.Wahlkreis',
       'ZweiterErstunterzeichnender.WahlkreisOrderBy'],
      dtype='object')

In [28]:
url_geschaefte = "https://www.gemeinderat-zuerich.ch/geschaefte/detailansicht-geschaeft"
df_deals_details_normalized['deals_details_url'] = url_geschaefte +'?gid='+ df_deals_details_normalized.gid

df_deals_details_normalized.head(2)
df_deals_details_normalized.columns


Index(['GeschaeftTitel', 'GeschaeftsJahr', 'GeschaeftsNummer', 'GeschaeftsArt',
       'Erstunterzeichnender', 'ZweiterErstunterzeichnender',
       'EinreichendeGruppe', 'PendentBei', 'FristBis', 'Departemente',
       'Ablaufschritte', 'VerknuepfteGeschaefte', 'gid',
       'Erstunterzeichnender.Id', 'Erstunterzeichnender.Name',
       'Erstunterzeichnender.Vorname', 'Erstunterzeichnender.Titel',
       'Erstunterzeichnender.Partei', 'Erstunterzeichnender.Wahlkreis',
       'Erstunterzeichnender.WahlkreisOrderBy',
       'ZweiterErstunterzeichnender.Id', 'ZweiterErstunterzeichnender.Name',
       'ZweiterErstunterzeichnender.Vorname',
       'ZweiterErstunterzeichnender.Titel',
       'ZweiterErstunterzeichnender.Partei',
       'ZweiterErstunterzeichnender.Wahlkreis',
       'ZweiterErstunterzeichnender.WahlkreisOrderBy', 'deals_details_url'],
      dtype='object')

In [29]:
df_deals_details_normalized.to_excel('deals_details_normalized.xlsx')  


In [30]:
#deals_details

So können z.B. die Ablaufschritte einzeln noch herausgelesen werden 

In [31]:
df_deals_details_normalized2 = pd.json_normalize(
    deals_details, 
    record_path=['Ablaufschritte'], 
    meta=[['Id', 'Name', 'Vorname']],
    meta_prefix='meta-',
    record_prefix='path-',
    errors='ignore' 
)



In [32]:
#deals_details
df_deals_details_normalized2.head(2)
#df_deals_details_normalized2.columns

,path-Datum,path-Created,path-Titel,path-Protokolleintrag,path-AblaufschrittDokumente,path-BeschlussNrGR,path-SitzungsNr,path-AudioLink,path-AudioLinkText,meta-Id.Name.Vorname
0,2022-04-06T00:00:00,2022-04-06T21:14:24.437,"Eingang, Frist 3 Monate","<p style=""margin: 0px 0px 0px 0px;text-align: ...","[{'Dokumenttitel': '2022_0126.pdf', 'Id': '68e...",5209.0,193.0,None,None,NaN
1,2022-03-23T00:00:00,2022-03-24T10:42:25.347,"Eingang, Frist 3 Monate","<p style=""margin: 0px 0px 0px 0px;text-align: ...","[{'Dokumenttitel': '2022_0109.pdf', 'Id': '4bd...",5143.0,191.0,None,None,NaN
